# OpenMM Notebook 1: Energy Minimization

This is a Jupyter notebook for running a simple example using the OpenMM engine in which we load structures, calculation energies and obtaining the minimum energy configuration.

For **more information about OpenMM** please see: 
https://openmm.org/


---

## Setting the environment for using OpenMM

We load the stardard scientific libraries plus the OpenMM python libraries, as shown in the OpenMM Manual (this included the system library *sys*):  
https://docs.openmm.org/latest/userguide/application/02_running_sims.html#a-first-example 

In [1]:
#Scientific (general) python libraries
import numpy as np
import matplotlib.pyplot as plt
import scipy.constants as ct

#OpenMM libraries
from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout

---

## Doing an OpenMM calculation

### 1.- Loading input files

We need **two essential input files**:  

- A *coordinate file* for the system that we want to simulate, with all atoms and their initial positions.  
- A **Force Field* file with the information required to calculate the forces between atoms and the energies of the system (see https://docs.openmm.org/latest/userguide/application/06_creating_ffs.html )

In this example both files are assumed to be in a subfolder called **input**.   

### 2.- Specify all details of the calculation
  
Once the two files are loaded, we need to create a *simulation system* which contains the information form these files and specifies the approximations to be employed and specify several details such as the approximations employed.



In [2]:
#Structure File with initial coordinates
coords = PDBFile("./input/butane-nonplanar.pdb")
#ForceField file with parameters for the energy and force calculations
model = ForceField("./input/butane.gaff2.xml")
# Specify aproximations employed and put all together
system = model.createSystem(coords.topology, nonbondedMethod=NoCutoff, constraints=HBonds)
integrator = VerletIntegrator(0.001 * picoseconds)
simulation = Simulation(coords.topology, system, integrator)
simulation.context.setPositions(coords.positions) #specify as initial positions those loaded in the pdb file

### 3.- Run the calculation

In this example we run an energy minimization (which normaly are being followed by subsequent dynamic calculations if desired).


In [3]:
#Give initial potential energy
initial_state = simulation.context.getState(getPositions=True,getEnergy=True)
Ep0=initial_state.getPotentialEnergy().value_in_unit(kilojoules_per_mole)
print(F"Potential energy: {Ep0:.2e} kJ/mol")

Potential energy: 3.89e+01 kJ/mol


In [4]:
# Run the simulation
print('Minimizing...')
simulation.minimizeEnergy(tolerance=0.1 * kilojoules_per_mole / nanometer,maxIterations=1000)

# Print final energy
final_state = simulation.context.getState(getPositions=True,getEnergy=True)
Epf=final_state.getPotentialEnergy().value_in_unit(kilojoules_per_mole)
print(F"Potential energy after minimization is {Epf:.2e}")


Minimizing...
Potential energy after minimization is 8.27e+00


In [5]:
#save the coordinates for final state
finalpositions = simulation.context.getState(getPositions=True).getPositions()
with open('./output/minimized-from-nonplanar.pdb', 'w') as f:
    app.PDBFile.writeFile(simulation.topology, finalpositions, f)

-------------------------   
**Question 1:**
How probable is this energy difference in thermal equilibrium at 300K, according to the Boltzmann factor?

   
**Question 2**: Repeat the calculation with another initial condition. Do you obtain the same minimum energy structure?